In [9]:
import random
import math
from sklearn.datasets import load_diabetes
import random
from typing import Optional

In [10]:
class Node:
    def __init__(self):
        self.feature: Optional[int] = None
        self.split: Optional[float] = None
        self.left: Optional["Node"] = None
        self.right: Optional["Node"] = None
        self.value: Optional[float] = None

In [24]:
class DecisionTree:
    def __init__(self, maxs=None, maxh=None, random_features=None):
        self.root = None
        self.maxs = maxs
        self.maxh = maxh
        self.random_features = random_features

    def build_tree(self, x: list[list[float]], y: list[float]):
        self.root = self.build(x,y,0)
    
    def build(self, x: list[list[float]], y: list[float], h:int):
        node = Node()
        n = len(x)
        if(n==0):
            return None

        if(self.maxh is not None):
            if(h >= self.maxh):
                node.value = sum(y)/len(y) #average
                return node
        
        if(self.maxs is not None):
            if(n<=self.maxs):
                node.value = sum(y)/len(y)
                return node
        
        d=len(x[0])
        l=[]

        for i in range(d):
            l.append(i)

        features=l

        if(self.random_features is not None): #using Random Forest
            features = random.sample(features, self.random_features)
        
        optj = None #optimal j
        opts = None #optimal s
        opte = None #optimal error
        optI_neg_x = None
        optI_neg_y = None
        optI_pos_x = None
        optI_pos_y = None

        for j in features:
            values=[]
            for ele in x:
                values.append(ele[j])

            for s in values:
                I_neg_x = []
                I_neg_y = []
                I_pos_x = []
                I_pos_y = []

            for i in range(n):
                if(x[i][j]<s):
                    I_neg_x.append(x[i])
                    I_neg_y.append(y[i])
                else:
                    I_pos_x.append(x[i])
                    I_pos_y.append(y[i])

            if((len(I_neg_y) == 0) or (len(I_pos_y) == 0)):
                continue

            neg_mean = (sum(I_neg_y)/len(I_neg_y))
            pos_mean = (sum(I_pos_y)/len(I_pos_y))

            err = 0

            for ele in I_neg_y:
                err = err + ((ele-neg_mean) ** 2)

            for ele in I_pos_y:
                err = err+ ((ele-pos_mean) ** 2)

            if((opte is None) or (err < opte)):
                    opte = err
                    optj = j
                    opts = s
                    optI_neg_x = I_neg_x
                    optI_neg_y = I_neg_y
                    optI_pos_x = I_pos_x
                    optI_pos_y = I_pos_y
            
        if(optj is None):
            node.value = sum(y)/len(y)
            return node
        
        lchild = self.build(optI_neg_x, optI_neg_y, h+1)
        rchild = self.build(optI_pos_x, optI_pos_y, h+1)

        node.feature = optj
        node.split = opts
        node.left = lchild
        node.right = rchild
        return node
        
    def predict(self, x):
        node = self.root
        while(node.value is None): #while we haven't reached a leaf
            if(x[node.feature]<node.split):
                node=node.left
            else:
                node=node.right
        return node.value
            





In [25]:
class random_forest:

    def __init__(self, num=10, maxs=None, maxh=None, random_features=None):
        self.num = num
        self.maxs = maxs
        self.maxh = maxh
        self.random_features = random_features
        self.trees = []

    def build_forest(self, x, y):
        n = len(x)
        for i in range(self.num):
            Dx = []
            Dy = []
            for i in range(n):
                rand = random.randint(0, n-1)
                Dx.append(x[rand])
                Dy.append(y[rand])

            tree = DecisionTree(self.maxs, self.maxh, self.random_features)
            tree.build_tree(Dx, Dy)
            self.trees.append(tree)

    def predict(self, x):
        predictions = []
        for ele in self.trees:
            predictions.append(ele.predict(x))

        avg = (sum(predictions)/len(predictions))
        return avg
    
    


In [26]:
def error_calc(t, x, y):
    sum = 0

    for i in range(len(x)):
        pred = t.predict(x[i])
        sum += ((pred - y[i])**2)

    sum = (sum/len(x))
    return sum

In [ ]:
random.seed(42)
data = load_diabetes()
x = [list(row) for row in data.data]
y = list(data.target)
data = list(zip(x, y))
random.shuffle(data)

split = int(0.8 * len(x))
x_train, y_train = x[:split], y[:split]
x_test, y_test = x[split:], y[split:]


tree = DecisionTree(maxs=3, maxh=8)
tree.build_tree(x_train, y_train)
err1 = error_calc(tree, x_test, y_test)

forest = random_forest(num=10, maxs=3, maxh=8, random_features=3)
forest.build_forest(x_train, y_train)
err2 = error_calc(forest, x_test, y_test)

forest = random_forest(num=50, maxs=3, maxh=8, random_features=3)
forest.build_forest(x_train, y_train)
err3 = error_calc(forest, x_test, y_test)

print(err1, err2, err3)


3875.9442421808 3415.3304347673934 3496.602233928104


In [ ]:
pri